In [1]:
!pip install -U transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 95.2 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


## Local Inference on GPU
Model page: https://huggingface.co/ras0k/whisper-rap-queb-v10

⚠️ If the generated code snippets do not work, please open an issue on either the [model repo](https://huggingface.co/ras0k/whisper-rap-queb-v10)
			and/or on [huggingface.js](https://github.com/huggingface/huggingface.js/blob/main/packages/tasks/src/model-libraries-snippets.ts) 🙏

In [2]:
# Use a pipeline as a high-level helper
from transformers import pipeline

pipe = pipeline("automatic-speech-recognition", model="ras0k/whisper-rap-queb-v10")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/1259 [00:00<?, ?it/s]

generation_config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

processor_config.json:   0%|          | 0.00/410 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/316 [00:00<?, ?B/s]

In [3]:
# Load model directly
from transformers import AutoProcessor, AutoModelForSpeechSeq2Seq

processor = AutoProcessor.from_pretrained("ras0k/whisper-rap-queb-v10")
model = AutoModelForSpeechSeq2Seq.from_pretrained("ras0k/whisper-rap-queb-v10")

Loading weights:   0%|          | 0/1259 [00:00<?, ?it/s]

In [5]:
import gradio as gr
from transformers import pipeline
import torch
import os

# 1. Check for GPU
device = "cuda:0" if torch.cuda.is_available() else "cpu"
print(f"Loading model on: {device}")

# 2. Initialize the pipeline WITH chunking enabled
pipe = pipeline(
    "automatic-speech-recognition",
    model="ras0k/whisper-rap-queb-v10",
    device=device,
    chunk_length_s=30,  # <-- Slices long audio into 30s chunks automatically
    batch_size=8,       # <-- Speeds up processing by running chunks in parallel
)

# 3. Define the transcription function for multiple files
def transcribe_multiple(audio_files):
    if not audio_files:
        return []

    output_txt_files = []

    # Loop through each uploaded file in sequence
    for audio_path in audio_files:
        # Extract the original filename and strip the extension (e.g., .mp3)
        base_name = os.path.basename(audio_path)
        file_root, _ = os.path.splitext(base_name)

        # Define the output .txt filename
        output_txt_path = f"{file_root}.txt"

        # Process the transcription
        print(f"Processing: {base_name}...")
        prediction = pipe(audio_path, return_timestamps=False)
        text = prediction["text"]

        # Write the transcription to a .txt file
        with open(output_txt_path, "w", encoding="utf-8") as f:
            f.write(text)

        # Add the generated .txt file to our output list
        output_txt_files.append(output_txt_path)

    print("All files processed successfully!")
    # Return the list of file paths so Gradio can offer them as downloads
    return output_txt_files

# 4. Build and launch the Gradio Interface
demo = gr.Interface(
    fn=transcribe_multiple,
    # Use gr.File to allow multiple audio uploads, defaulting to the file explorer
    inputs=gr.File(file_count="multiple", file_types=["audio"], label="Upload Audio Files (.mp3, .wav, etc.)"),
    # Use gr.File for the output so the user can download the generated .txt files
    outputs=gr.File(label="Download Transcriptions (.txt)"),
    title="🎙️ Whisper Rap Queb v5",
    description="Upload multiple audio files! The pipeline will automatically process them in sequence and generate a .txt file for each."
)

demo.launch(debug=True)

Loading model on: cuda:0


Loading weights:   0%|          | 0/1259 [00:00<?, ?it/s]

Using `chunk_length_s` is very experimental with seq2seq models. The results will not necessarily be entirely accurate and will have caveats. More information: https://github.com/huggingface/transformers/pull/20104. Ignore this warning with pipeline(..., ignore_warning=True). To use Whisper for long-form transcription, use rather the model's `generate` method directly as the model relies on it's own chunking mechanism (cf. Whisper original paper, section 3.8. Long-form Transcription).


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://605efbef2cc15474f6.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


ERROR:    Exception in ASGI application
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/uvicorn/protocols/http/httptools_impl.py", line 416, in run_asgi
    result = await app(  # type: ignore[func-returns-value]
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/uvicorn/middleware/proxy_headers.py", line 60, in __call__
    return await self.app(scope, receive, send)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/fastapi/applications.py", line 1160, in __call__
    await super().__call__(scope, receive, send)
  File "/usr/local/lib/python3.12/dist-packages/starlette/applications.py", line 107, in __call__
    await self.middleware_stack(scope, receive, send)
  File "/usr/local/lib/python3.12/dist-packages/starlette/middleware/errors.py", line 186, in __call__
    raise exc
  File "/usr/local/lib/python3.12/dist-packages/starlette/middleware/error

Processing: 001-Django-s-o-le-flem.mp3...
Processing: 002-Kaytranada-glowed-up.mp3...
Processing: 003-Loud-toutes-les-femmes-savent-danser.mp3...
Processing: 004-Freeze-corleone-dans-les-buissons.mp3...
Processing: 005-Jeune-loup-sensuelle.mp3...
Processing: 006-Loud-ttttt.mp3...
Processing: 007-Loud-devenir-immortel-et-puis-mourir.mp3...
Processing: 008-Loud-nouveaux-riches.mp3...
Processing: 009-Roi-heenok-38-special.mp3...
Processing: 010-Fouki-gaye.mp3...
All files processed successfully!
Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://605efbef2cc15474f6.gradio.live


In [6]:
import zipfile
import glob
import os

# Find all .txt files in the current directory
txt_files = glob.glob("*.txt")

# Define the name for the output zip file
zip_filename = "transcriptions.zip"

# Create a zip file and add all .txt files to it
with zipfile.ZipFile(zip_filename, 'w') as zipf:
    for file in txt_files:
        zipf.write(file, os.path.basename(file))

print(f"Successfully created {zip_filename} containing {len(txt_files)} transcription files.")

Successfully created transcriptions.zip containing 10 transcription files.
